In [1]:
# This script extracts all TRF results from pickle files into a single CSV file

import numpy as np
from pathlib import Path
import pickle
import csv

# Define the paths that will be used throughout
DATA_ROOT = Path('/Users/zorkabozilovic/Desktop/PART1')
old_dir = DATA_ROOT / 'not improved results'
new_dir = DATA_ROOT / 'results'

bands = ['delta', 'theta', 'alpha', 'beta', 'wideband']

# One row per subject per band per pipeline per combo (20*5*2*36 = 7200 rows)
rows = []

# Loop through both pipelines, all 20 subjects and all 5 bands
for pipeline, results_dir in [('not standardized', old_dir), ('standardized', new_dir)]:
    
    for sub in range(1, 21):
        
        # Specify the group
        group = 'Musician' if sub >= 11 else 'Non-musician'
        
        # Specify the band
        for band in bands:
            fpath = results_dir / f'Sub{sub}_{band}.pkl'
            if not fpath.exists():
                continue
            with open(fpath, 'rb') as f:
                data = pickle.load(f)

            # For the specific pipeline, subject and band add results to rows for both encoding and decoding
            
            # Encoding
            for combo, res in data['encode_results'].items():
                if 'error' in res:
                    continue
                
                # Specify the number of features
                n_feats = len(combo.split('+'))

                # Get correlation coefficient per trail (r's)
                trial_rs = res['trial_rs']

                # Get proportion explained per trial, averaging across 64 sensors  (was not used in the thesis, but was left for future research)
                trial_pe = []
                for t in range(len(trial_rs)):
                    pe = res['trials'][f'trial{t}']['proportion_explained']
                    trial_pe.append(float(np.mean(pe.x))) 

                # Adding all columns for one row
                row = {
                    'subject': f'Sub{sub}',
                    'group': group,
                    'band': band,
                    'direction': 'encoding',
                    'pipeline': pipeline,
                    'combo_name': combo,
                    'n_features': n_feats,
                    'mean_r': np.mean(trial_rs), # Average across all trails
                    'mean_prop_explained': np.mean(trial_pe), # Average across all trails  (was not used in the thesis, but was left for future research)
                    't_run': res['model'].t_run, # Boosting fit time in seconds
                }
                # Add individual trial values (for r's and pe's) as separate columns
                for i, r in enumerate(trial_rs):
                    row[f'r_trial_{i+1}'] = r
                for i, pe in enumerate(trial_pe): # (was not used in the thesis, but was left for future research)
                    row[f'pe_trial_{i+1}'] = pe
                rows.append(row)

                
            # Decoding
            for feat, res in data['decode_results'].items():
                if 'error' in res:
                    continue

                # Get correlation coefficient per trail (r's)
                trial_rs = res['trial_rs']
                
                # Get proportion explained per trial
                trial_pe = []
                for t in range(len(trial_rs)):
                    pe = res['trials'][f'trial{t}']['proportion_explained']
                    if hasattr(pe, 'x'):
                        trial_pe.append(float(np.mean(pe.x))) # Mel has 8 bands
                    else:
                        trial_pe.append(float(pe))
                    
                # Adding all columns for one row
                row = {
                    'subject': f'Sub{sub}',
                    'group': group,
                    'band': band,
                    'direction': 'decoding',
                    'pipeline': pipeline,
                    'combo_name': feat,
                    'n_features': 1, # Decoding only has one feature at a time
                    'mean_r': np.mean(trial_rs), # Average across all trails
                    'mean_prop_explained': np.mean(trial_pe), # Average across all trails
                    't_run': res['model'].t_run, # Boosting fit time in seconds
                }
                # Add individual trial values (for r's and pe's) as separate columns
                for i, r in enumerate(trial_rs):
                    row[f'r_trial_{i+1}'] = r
                for i, pe in enumerate(trial_pe):
                    row[f'pe_trial_{i+1}'] = pe
                rows.append(row)

# Make the CSV file
out_path = DATA_ROOT / 'all_trf_results.csv'
fieldnames = list(rows[0].keys())
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f'Saved {len(rows)} rows to {out_path}')

Saved 7200 rows to /Users/zorkabozilovic/Desktop/PART1/all_trf_results.csv
